# MRMR 2027: generated disease maps

The MRMR experiments use two hand-drawn TYLCV maps, `mrmr-clustered-100.png` and `mrmr-notclustered-100.png`. This notebook experiments with generating such maps from the disease spreading model of the simulator (`EpidemicSpreadEnvironment`), instead of drawing them:

- the maps come in **pairs**: a clustered and an unclustered version with **exactly the same number of diseased plants**;
- the pairs are generated in several **sizes**: 30x30, 100x100, and the new 200x200 for scenarios with more robots;
- everything is determined by the provided random **seed**.

For the time being, the notebook only generates and visualizes the maps; it does not save them or register new environments.

In [ ]:
import logging
import pathlib
import sys
sys.path.append("../..")

import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import disease_maps
from disease_maps import cluster_stats, diseased_count, generate_disease_map
logging.getLogger().setLevel(logging.WARNING)  # the epidemic model logs every day

In [ ]:
seed = 1                    # all the maps are determined by this seed
sizes = [30, 100, 200]      # the maps are size x size cells
disease_fraction = 0.05     # the fraction of diseased plants, the same in both maps of a pair
# the generation parameters of the two versions, by default those of disease_maps.py
clustered = dict(disease_maps.CLUSTERED)
unclustered = dict(disease_maps.UNCLUSTERED)
# the seeds of the seed-dependence experiment, and its map size
compare_seeds = [1, 2, 3]
compare_size = 100

## The generation model

Both versions run the epidemic model of the simulator day by day, until the diseased plants (infected or destroyed, value < 1.0) reach the target `disease_fraction * size * size`:

- **Clustered:** the default number of initial infections for the size (`default_infection_seeds`), spreading through the default neighborhood (`default_spread_dimension`) with a high transmission probability. The disease grows in compact outbreaks around the initial infections.
- **Unclustered:** a weak spread to the immediate neighbors (a 3x3 neighborhood), plus *long jumps*: every day, new infections at randomly chosen healthy plants, for instance carried by flying insects. The disease is scattered over many small spots.

In the parameters, the fractional values of `infection-seeds` and `jumps-per-day` are relative to the target number of diseased plants.

The last day usually overshoots the target. The surplus is removed from the newest infections, in random order, so both maps of a pair have exactly the target number of diseased plants.

Every map has its own random generator, derived from `(seed, width, height, version)`. A map therefore depends only on the seed and on its own size and version, and not on which other maps are generated.

The generator is `generate_disease_map` in `src/disease_maps.py`. Environments use it through `tylcv-generated` in their exp/run, and the MRMR experiments run on the generated 100x100 pair (`environment/mrmr-generated-{clustered,unclustered}-100`). See `DESIGN-ClusteredDiseaseMap.md`.

In [ ]:
VERSIONS = [("clustered", clustered), ("unclustered", unclustered)]


def generate_map(size, seed, version, params, fraction):
    """A size x size generated map: the epidemic model and the days it ran"""
    return generate_disease_map(size, size, version, seed, fraction, params=params)


def show_map(ax, value, title):
    """A disease map: healthy white, infected gray, destroyed black; x to the right, y up"""
    ax.imshow(value.T, origin="lower", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(title, fontsize=10)

## The pairs in all sizes

The number of diseased plants is the same in both maps of a pair. The spot statistics show the difference in clustering: the clustered maps have a few large spots, and the unclustered ones many small spots. Spots are connected groups of diseased plants, including diagonal neighbors.

In [ ]:
maps = {}
rows = []
for size in sizes:
    for name, params in VERSIONS:
        env, days = generate_map(size, seed, name, params, disease_fraction)
        maps[(size, name)] = env.value
        rows.append({"size": f"{size}x{size}", "version": name, "days": days, **cluster_stats(env.value)})
pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(2, len(sizes), figsize=(4 * len(sizes), 8))
for col, size in enumerate(sizes):
    for row, name in enumerate(["clustered", "unclustered"]):
        value = maps[(size, name)]
        show_map(axes[row, col], value, f"{name} {size}x{size}: {diseased_count(value)} diseased")
fig.suptitle(f"Generated disease map pairs, seed {seed}, {disease_fraction:.0%} diseased")
fig.tight_layout()

## Comparison with the hand-drawn maps

The hand-drawn maps of the MRMR experiments, next to the generated 100x100 pair. In the hand-drawn pair, the two maps have different amounts of disease.

In [ ]:
reference_dir = pathlib.Path("../../../data/expruns/environment")
reference = {}
for name, filename in [("clustered", "mrmr-clustered-100.png"), ("unclustered", "mrmr-notclustered-100.png")]:
    picture = iio.imread(reference_dir / filename)
    # as loaded by the simulator: the first channel divided by 255, indexed as [x, y]
    reference[name] = (picture[:, :, 0] if picture.ndim == 3 else picture) / 255.0
rows = [{"map": f"hand-drawn {name}", **cluster_stats(value)} for name, value in reference.items()]
if 100 in sizes:
    rows += [{"map": f"generated {name}", **cluster_stats(maps[(100, name)])} for name in ["clustered", "unclustered"]]
display(pd.DataFrame(rows))

panels = [(f"hand-drawn {name}", value) for name, value in reference.items()]
if 100 in sizes:
    panels += [(f"generated {name}", maps[(100, name)]) for name in ["clustered", "unclustered"]]
fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
for ax, (title, value) in zip(axes, panels):
    show_map(ax, value, f"{title}: {diseased_count(value)}")
fig.tight_layout()

## Dependence on the seed

The same size with different seeds gives different maps with the same number of diseased plants. Generating a map twice with the same seed gives the same map.

In [ ]:
for name, params in VERSIONS:
    first, _ = generate_map(compare_size, compare_seeds[0], name, params, disease_fraction)
    again, _ = generate_map(compare_size, compare_seeds[0], name, params, disease_fraction)
    assert np.array_equal(first.value, again.value), f"{name} is not determined by the seed"
print("Every map is determined by its seed")

fig, axes = plt.subplots(2, len(compare_seeds), figsize=(4 * len(compare_seeds), 8))
for col, s in enumerate(compare_seeds):
    for row, (name, params) in enumerate(VERSIONS):
        env, _ = generate_map(compare_size, s, name, params, disease_fraction)
        show_map(axes[row, col], env.value, f"{name} {compare_size}x{compare_size}, seed {s}")
fig.tight_layout()